# DAM405 Practical 4 - AutoML with FLAML
Run the cells top to bottom. **Take one screenshot per task** (see README).

## Task 1 - Install and load data

In [ ]:
!pip install -q "flaml[automl]" scikit-learn joblib

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(X.shape, X_train.shape, X_test.shape)

## Task 2 - Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time
t0 = time.perf_counter()
base = LogisticRegression(max_iter=2000).fit(X_train, y_train)
base_time = time.perf_counter() - t0
base_acc = accuracy_score(y_test, base.predict(X_test))
print('Baseline accuracy:', round(base_acc, 4))

## Task 3 - Run AutoML

In [ ]:
from flaml import AutoML

automl = AutoML()
t0 = time.perf_counter()
automl.fit(X_train, y_train, task='classification',
           metric='accuracy', time_budget=60)  # increase for a better search
auto_time = time.perf_counter() - t0
print(f'AutoML search finished in {auto_time:.1f} s')

## Task 4 - Inspect the best model

In [ ]:
print('Best estimator :', automl.best_estimator)
print('Best config    :', automl.best_config)
print('Best CV score  :', 1 - automl.best_loss)

## Task 5 - Evaluate on the test set

In [ ]:
auto_acc = accuracy_score(y_test, automl.predict(X_test))
print('AutoML test accuracy  :', round(auto_acc, 4))
print('Baseline test accuracy:', round(base_acc, 4))
print(f'Extra compute: baseline {base_time:.2f}s vs AutoML {auto_time:.1f}s')

## Task 6 - Save the best model

In [ ]:
import joblib
joblib.dump(automl, 'automl_model.joblib')
print('Saved automl_model.joblib')
!ls -lh automl_model.joblib

## Optional - larger budget / different metric
Change `time_budget` (e.g. 120) or `metric='roc_auc'` above and re-run Tasks 3-5, noting how the result changes.